# 01 — Ingesta y extracción de documentos PDF

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Prototipado del pipeline de ingesta

---

Este notebook cubre la primera fase del pipeline RAG: la carga, extracción
y estructuración del contenido de un documento PDF científico de química
mediante **Docling** (IBM).

El output de este notebook es la entrada del notebook `02_embeddings.ipynb`.

In [1]:
"""
Notebook: 01_ingesta.ipynb

Objetivo:
    Cargar un documento PDF científico de química y extraer su contenido
    estructurado (texto, tablas y elementos especiales) usando Docling (IBM),
    tal y como se haría en la fase de ingesta de un pipeline RAG en producción.

    Este notebook se centra exclusivamente en la carga, extracción y
    validación inicial del documento. El chunking y la vectorización
    se abordan en los notebooks posteriores.

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2025-04-29
Versión: 1.0.0
"""

'\nNotebook: 01_ingesta.ipynb\n\nObjetivo:\n    Cargar un documento PDF científico de química y extraer su contenido\n    estructurado (texto, tablas y elementos especiales) usando Docling (IBM),\n    tal y como se haría en la fase de ingesta de un pipeline RAG en producción.\n\n    Este notebook se centra exclusivamente en la carga, extracción y\n    validación inicial del documento. El chunking y la vectorización\n    se abordan en los notebooks posteriores.\n\nFuente de datos:\n    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,\n    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical\n    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)\n    complexes, with apoptosis-inducing properties in cisplatin-resistant\n    neuroblastoma cells. Frontiers in Chemistry. 2024.\n    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/\n\n    Documento utilizado exclusivamente con fines de investigación y desarrollo.\n    No se distrib

## 1. Configuración del entorno

In [2]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [3]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [4]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.113+


## 2. Instalación de dependencias

In [5]:
# Instalación de Docling (IBM) para extracción estructurada de PDFs científicos
# Se instala en modo silencioso (-q) para reducir el output en Colab
%pip install docling -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 73.6 MB/s eta 0:00:00
   ━━

In [6]:
# Verificación de la instalación de Docling
from docling.document_converter import DocumentConverter

print("Docling importado correctamente")

Docling importado correctamente


## 3. Definición de rutas

In [7]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Ruta al directorio de documentos de entrada
DIR_RAW = PROYECTO_RAIZ / 'data' / 'raw'

# Ruta al directorio de salida para artefactos del notebook
DIR_OUTPUT = PROYECTO_RAIZ / 'data' / 'processed'
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# Nombre del documento de prueba
NOMBRE_PDF = 'PMC10967698.pdf'
RUTA_PDF = DIR_RAW / NOMBRE_PDF

# Validación de existencia del archivo antes de continuar
assert RUTA_PDF.exists(), (
    f"Archivo no encontrado: {RUTA_PDF}\n"
    f"Asegúrate de que el PDF está en: {DIR_RAW}"
)

print(f"Documento localizado : {RUTA_PDF}")
print(f"Tamaño del archivo   : {RUTA_PDF.stat().st_size / 1024:.1f} KB")

Documento localizado : /content/drive/MyDrive/chem-rag-assistant/data/raw/PMC10967698.pdf
Tamaño del archivo   : 785.0 KB


## 4. Extracción del documento con Docling

In [8]:
# Inicialización del convertidor de Docling
# DocumentConverter detecta automáticamente el tipo de documento
# y aplica el pipeline de extracción adecuado
converter = DocumentConverter()

print("Iniciando extracción del documento...")
resultado = converter.convert(str(RUTA_PDF))
print("Extracción completada")

Iniciando extracción del documento...


[INFO] 2026-04-30 06:19:03,626 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 06:19:03,638 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 06:19:03,649 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 06:19:05,030 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-30 06:19:05,809 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 06:19:05,816 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 06:19:07,402 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 06:19:07,406 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 06:19:07,410 [RapidOCR] download_file.py:68: Initiating download: https

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Extracción completada


In [9]:
# Exportación del documento extraído a formato Markdown
# Markdown preserva la estructura del documento (títulos, tablas, listas)
# y es el formato óptimo como entrada para el pipeline de chunking
documento_md = resultado.document.export_to_markdown()

print(f"Caracteres extraídos : {len(documento_md):,}")
print(f"Palabras aproximadas : {len(documento_md.split()):,}")

Caracteres extraídos : 59,614
Palabras aproximadas : 10,851


## 5. Validación del contenido extraído

In [10]:
# Inspección de los primeros 2000 caracteres del documento extraído
# para verificar que la extracción preserva la estructura correctamente
CARACTERES_PREVIEW = 2000

print("=" * 60)
print("PREVIEW DEL DOCUMENTO EXTRAÍDO")
print("=" * 60)
print(documento_md[:CARACTERES_PREVIEW])
print("...")
print("=" * 60)

PREVIEW DEL DOCUMENTO EXTRAÍDO
<!-- image -->

## RSC Advances

## PAPER

<!-- image -->

Cite this: RSC Adv. , 2024, 14 , 10244

Received 16th February 2024 Accepted 14th March 2024

DOI: 10.1039/d4ra01195c

rsc.li/rsc-advances

## Introduction

N-heterocyclic carbenes (NHCs),  rst described in 1991, 1 have found many applications. 2 There are several structural features that allow the tuning of their electronic properties. Ring size, the adjacent heteroatoms, N -substituents, and the backbone can be modi  ed. Changing one or more structural properties of a NHC ligand can lead to signi  cantly di ff erent reactivities and stabilities of the resulting complexes. 3 O  en several NHC units are combined in multidentate ligands, making use of the chelating e ff ect, and a plethora of multidentate NHC metal complexes has been reported. 4,5

a Technical University of Munich, School of Natural Sciences, Department of Chemistry and Catalysis Research Center, Molecular Catalysis, Lichtenber

In [11]:
# Validación de presencia de términos clave del paper
# Verificamos que Docling extrajo correctamente la terminología
# organometálica (compuestos, metales, técnicas analíticas)
TERMINOS_CLAVE = [
    'Pd',            # Paladio
    'Pt',            # Platino
    'Au',            # Oro
    'NHC',           # N-Heterocyclic Carbene
    'cisplatin',     # Referencia farmacológica
    'apoptosis',     # Mecanismo biológico
    'neuroblastoma', # Línea celular estudiada
]

print("Validación de términos clave en el documento extraído:")
print("-" * 45)
for termino in TERMINOS_CLAVE:
    encontrado = termino.lower() in documento_md.lower()
    estado = "OK" if encontrado else "NO ENCONTRADO"
    print(f"  [{estado}]  {termino}")
print("-" * 45)

Validación de términos clave en el documento extraído:
---------------------------------------------
  [OK]  Pd
  [OK]  Pt
  [OK]  Au
  [OK]  NHC
  [OK]  cisplatin
  [OK]  apoptosis
  [OK]  neuroblastoma
---------------------------------------------


## 6. Persistencia del documento procesado

In [12]:
# Guardado del documento extraído en formato Markdown
# Este archivo será la entrada del notebook 02_embeddings.ipynb
RUTA_OUTPUT_MD = DIR_OUTPUT / 'PMC10967698_extracted.md'

with open(RUTA_OUTPUT_MD, 'w', encoding='utf-8') as f:
    f.write(documento_md)

print(f"Documento guardado en : {RUTA_OUTPUT_MD}")
print(f"Tamaño del archivo    : {RUTA_OUTPUT_MD.stat().st_size / 1024:.1f} KB")

Documento guardado en : /content/drive/MyDrive/chem-rag-assistant/data/processed/PMC10967698_extracted.md
Tamaño del archivo    : 58.6 KB


## 7. Resumen de la ejecución

In [13]:
# Resumen final del proceso de ingesta
print("=" * 60)
print("RESUMEN — INGESTA COMPLETADA")
print("=" * 60)
print(f"  Documento origen  : {NOMBRE_PDF}")
print(f"  Caracteres        : {len(documento_md):,}")
print(f"  Palabras aprox.   : {len(documento_md.split()):,}")
print(f"  Output generado   : {RUTA_OUTPUT_MD.name}")
print("=" * 60)
print("Siguiente paso: 02_embeddings.ipynb")

RESUMEN — INGESTA COMPLETADA
  Documento origen  : PMC10967698.pdf
  Caracteres        : 59,614
  Palabras aprox.   : 10,851
  Output generado   : PMC10967698_extracted.md
Siguiente paso: 02_embeddings.ipynb
